In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import os
from pathlib import Path
import statsmodels.formula.api as smf

In [2]:
ROOT = Path("/home/hanwenying")
DATADIR = ROOT / "rothman-data/transcriptlength"

In [3]:
tpm = pd.read_parquet(DATADIR / "tpm-gtex-v11.parquet", engine='pyarrow').reset_index()

In [ ]:
tpm.head()

In [ ]:
tpm = tpm.set_index(tpm['Description'])
tpm = tpm.drop(['Name', 'Description'], axis=1)
tpm = tpm.T
tpm = tpm.reset_index()
tpm = tpm.rename_axis(None, axis=1)
tpm = tpm.rename(columns={'index':'SAMPLE'})

In [13]:
metadata = pd.read_csv(DATADIR / "metadata.tsv", sep='\t', dtype=str)

In [48]:
metadata.shape, tpm.shape

((19788, 5), (19788, 74629))

In [ ]:
agemap = {'20-29':0, '30-39':1, '40-49':2, '50-59':3, '60-69':4, '70-79':5}

In [47]:
wash7p = tpm[['SAMPLE', 'WASH7P']]

In [49]:
wash7p_dm = pd.merge(wash7p, metadata, on='SAMPLE', how='inner')

In [60]:
wash7p_dm['AGE'] = wash7p_dm['AGE'].map(agemap)

In [62]:
wash7p_dm['SEX'] =wash7p_dm['SEX'].astype(int) - 1

In [63]:
wash7p_dm

,SAMPLE,WASH7P,SUBJID,SEX,AGE,TISSUE
0,GTEX-1117F-0005-SM-HL9SH,1.163903,GTEX-1117F,1,4,0005
1,GTEX-1117F-0011-R10b-SM-GI4VE,1.260954,GTEX-1117F,1,4,0011
2,GTEX-1117F-0011-R11b-SM-GIN8R,3.023480,GTEX-1117F,1,4,0011
3,GTEX-1117F-0011-R2b-SM-GI4VL,0.335984,GTEX-1117F,1,4,0011
4,GTEX-1117F-0011-R3a-SM-GJ3PJ,0.631406,GTEX-1117F,1,4,0011
...,...,...,...,...,...,...
19783,GTEX-ZZPU-2326-SM-GOQYU,3.870932,GTEX-ZZPU,1,3,2326
19784,GTEX-ZZPU-2426-SM-5E44I,1.475226,GTEX-ZZPU,1,3,2426
19785,GTEX-ZZPU-2526-SM-GOQZ3,6.310629,GTEX-ZZPU,1,3,2526
19786,GTEX-ZZPU-2626-SM-5E45Y,0.703600,GTEX-ZZPU,1,3,2626


In [64]:
model = smf.mixedlm(
            "WASH7P ~ AGE + SEX + TISSUE", 
            wash7p_dm, 
            groups=wash7p_dm['SUBJID']
        )

In [65]:
res = model.fit()

In [68]:
t, p = res.tvalues['AGE'], res.pvalues['AGE']

In [ ]:
t,p

(np.float64(0.3542308516510685), np.float64(0.723165884362958))

: 